In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
sys.path.append('..')

import utilities.functions as functions
import utilities.descritiva as descritiva

from utilities.descritiva import  (
    matriz_migracao,
    cria_base_decil_wide
)

from utilities.functions import (
    summary,
    gerar_resumo_decis,
    resumo_coorte_ativa,
    process_orders_pandas

)
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 


In [ ]:
df_pub_u = pd.read_parquet(BASE_PATH / "silver" / "df_p_2.parquet")

In [ ]:
#df_pub_u = pd.read_parquet(BASE_PATH / "gold" / "df_pub_un.parquet")
df_pub_u=process_orders_pandas(df_pub_u)

In [ ]:
df_pub_u = df_pub_u[["customer_id","is_target", "order_created_month", "num_pedidos_mes", "num_pedidos_hist",'total_amount_mes','ticket_medio']].drop_duplicates().reset_index(drop=True)
df_pub_u['pedidos_sum'] = np.where(
        df_pub_u['num_pedidos_mes'] > 10, '10+',
        df_pub_u['num_pedidos_mes'].astype(str)
    )
df_pub_u.head()

In [ ]:
df_pub_u.head()

Sumarizacao na visao cliente - mes

In [ ]:
summary(df_pub_u)

Clientes mes x target

 Matriz de migracao - cliente

In [ ]:
matriz_migracao(df_pub_u,mes_0=12,mes_1=1)

Dado o montante de clientes novos em janeiro a analise sera feita basead em cliente ambos os meses e cliente somente em janeiro

In [ ]:
id_both_monht=df_pub_u[df_pub_u['order_created_month']==12]['customer_id'].unique()
publico_janeiro_dezembro = df_pub_u[df_pub_u['customer_id'].isin(id_both_monht)].reset_index(drop=True)
publico_janeiro = df_pub_u[~df_pub_u['customer_id'].isin(id_both_monht)].reset_index(drop=True)

In [ ]:
publico_janeiro_dezembro.to_parquet(BASE_PATH / "gold" / "publico_janeiro_dezembro.parquet", index=False)

Pedidos por mes - target

In [ ]:
pedidos_total = (
    publico_janeiro_dezembro.groupby(['is_target','order_created_month'])['num_pedidos_mes']
      .sum()
      .reset_index(name='numero_de_pedidos_total')
)

pedidos_total['pct_mes'] = pedidos_total.groupby('order_created_month')['numero_de_pedidos_total'] \
                 .transform(lambda x: x / x.sum() * 100).round(2)

pedidos_total


Distruibuicao do total amount por mes

In [ ]:
df_stats_mes = publico_janeiro_dezembro.groupby(['order_created_month', 'is_target'])['total_amount_mes'].agg(
    Média=('mean'),
    Mediana=('median'),
    Mínimo=('min'),
    Máximo=('max'),
    Desvio_Padrão=('std')
).round(2)
df_stats_mes

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(
    data=publico_janeiro_dezembro,
    y='total_amount_mes',
    x='order_created_month',  
    hue='is_target'          
)
plt.title('Distribuição total amount menssal x  e  target')
plt.ylabel('Total Amount (R$)')
plt.xlabel('Mês')
plt.legend(title='Grupo')
plt.show()

Observa-se grande numero de outliers, verificar a distribuicao removendo outliers.

In [ ]:
print(publico_janeiro_dezembro[publico_janeiro_dezembro['order_created_month']==12]['total_amount_mes'].describe().round(2))
print(publico_janeiro_dezembro[publico_janeiro_dezembro['order_created_month']==1]['total_amount_mes'].describe().round(2))



In [ ]:
red_square = dict(markerfacecolor='r', markeredgecolor='r', marker='.')
publico_janeiro_dezembro['total_amount_mes'].plot(kind='box', xlim=(0, 5500), vert=False, flierprops=red_square, figsize=(16,2))

In [ ]:
p75=np.nanpercentile(publico_janeiro_dezembro.loc[publico_janeiro_dezembro['order_created_month'] == 12, 'total_amount_mes' ],75)
p25=np.nanpercentile(publico_janeiro_dezembro.loc[publico_janeiro_dezembro['order_created_month'] == 12, 'total_amount_mes' ],25)

In [ ]:
distance = 1.5 * (p75 - p25)
lim_sup= p75+distance
lim_inf = p25-distance
print(lim_sup)


Marcacao de outliers - amount na base historica e griando uma base somente com outliers

In [ ]:
publico_janeiro_dezembro['outlier'] = publico_janeiro_dezembro['total_amount_mes'] > lim_sup
df_out= publico_janeiro_dezembro[publico_janeiro_dezembro['total_amount_mes'] > lim_sup]

In [ ]:
total_amount = publico_janeiro_dezembro.groupby(['order_created_month', 'is_target']).agg(
    total_amount=('total_amount_mes', 'sum'),
    total_pedidos=('num_pedidos_mes', 'sum')
).round(2)

total_amount_out = df_out.groupby(['order_created_month', 'is_target']).agg(
    total_amount=('total_amount_mes', 'sum'),
    total_pedidos=('num_pedidos_mes', 'sum')
).round(2)
#total_amount_out
#total_amount

merged = (
    total_amount
    .merge(total_amount_out, on=['order_created_month', 'is_target'], how='left', suffixes=('_all', '_out'))
    .assign(
        pct_amount_out=lambda x: (x['total_amount_out'] / x['total_amount_all']).round(4),
        pct_pedidos_out=lambda x: (x['total_pedidos_out'] / x['total_pedidos_all']).round(4)
    )
)
merged

Migracao based in marcacao de outlier

In [ ]:
matriz_migracao(publico_janeiro_dezembro, 12, 1, group_by_extra='outlier')

Entender quem sao esse clientes outliers

In [ ]:
df_out

In [ ]:
decil_dict = cria_base_decil_wide(publico_janeiro_dezembro, mes_0=12)

# 2) Aplicar
publico_janeiro_dezembro['decil'] = publico_janeiro_dezembro['total_amount_mes'].apply(
    lambda x: max([decil for decil, (min_val, max_val) in decil_dict.items() if x >= min_val])
)

In [ ]:
clientes_mes = (
    publico_janeiro_dezembro.groupby(['decil'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

In [ ]:
publico_janeiro_dezembro

In [ ]:
def calcular_stats(group):
    media = group['total_amount_mes'].mean()
    std = group['total_amount_mes'].std()
    return pd.Series({
        'Total_Clientes': group['customer_id'].nunique(),
        'Total_Orders': group['num_pedidos_mes'].sum(),  # ou count() dependendo da sua estrutura
        'Média_Amount': media,
        'Mediana_Amount': group['total_amount_mes'].median(),
        'Mínimo_Amount': group['total_amount_mes'].min(),
        'Máximo_Amount': group['total_amount_mes'].max(),
        'Desvio_Padrão_Amount': std,
        'CV_Amount': (std / media * 100) if media != 0 else np.nan
    })

df_stats_mes = publico_janeiro_dezembro.groupby(['order_created_month', 'is_target','decil']).apply(calcular_stats).round(2)
#df_stats_mes

In [ ]:
publico_janeiro_dezembro.to_parquet(BASE_PATH / "gold" / "publico_janeiro_dezembro.parquet", index=False)

In [ ]:
os.makedirs('../../Resultados', exist_ok=True)

#decil_summary.to_csv('../../Resultados/decil_summary.csv', index=False)



Numero de pedidos em Janeiro e Dezembro para o quem esteve em ambos meses

In [ ]:
publico_janeiro_dezembro

In [ ]:

df_stats_mes = publico_janeiro_dezembro.groupby(['order_created_month', 'pedidos_sum']).agg(
    total_clientes=('customer_id', 'nunique')
).round(2)

df_stats_mes['pct_mes'] = df_stats_mes.groupby('order_created_month')['total_clientes'] \
                 .transform(lambda x: x / x.sum() * 100).round(2)


df_stats_hist = publico_janeiro_dezembro.groupby(['pedidos_sum']).agg(
    total_clientes=('customer_id', 'nunique')
).round(2)

df_stats_hist['pct_total'] = (df_stats_hist['total_clientes'] / df_stats_hist['total_clientes'].sum() * 100).round(2)

In [ ]:
publico_janeiro_dezembro.head()

In [ ]:
df_stats_mes

In [ ]:
publico_janeiro_dezembro.head(2)

In [ ]:
# Gráfico 1 - Total de clientes por mês e grupo
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)  # 1 linha, 2 colunas, posição 1
sns.barplot(
    data=df_stats_mes[df_stats_mes['order_created_month']==12],
    x='pedidos_sum',
    y='ticket_medio',
    hue='is_target'
)
plt.title('Número de Clientes por Mês')
plt.ylabel('Total de Clientes')
plt.xlabel('Mês')



In [ ]:
publico_janeiro_dezembro.head()

In [ ]:
publico_janeiro_dezembro

In [ ]:
publico_janeiro_dezembro.head()

In [ ]:
publico_janeiro_dezembro[
        (publico_janeiro_dezembro['num_pedidos_mes'] < 11) & 
        (publico_janeiro_dezembro['outlier'] == False)]

In [ ]:
total_amount = (
    publico_janeiro_dezembro[
        (publico_janeiro_dezembro['num_pedidos_mes'] < 11) & 
        (publico_janeiro_dezembro['outlier'] == False)
    ]
    .groupby(['order_created_month', 'is_target'])
    .agg(
        total_amount=('total_amount_mes', 'sum'),
        total_pedidos=('num_pedidos_mes', 'sum')
    )
    .round(2)
)
total_amount

In [ ]:
# CORREÇÃO: Filtro com parênteses corretos
total_amount_f= (
    publico_janeiro_dezembro[
        (publico_janeiro_dezembro['num_pedidos_mes'] < 11) & 
        (publico_janeiro_dezembro['outlier'] == False)
    ]
    .groupby(['order_created_month', 'is_target'])
    .agg(
        total_amount=('total_amount_mes', 'sum'),
        total_pedidos=('num_pedidos_mes', 'sum'),
        total_customer=('customer_id', 'nunique'),

    )
    .round(2)
)

total_amount_total = (
    publico_janeiro_dezembro.groupby(['order_created_month', 'is_target'])
    .agg(
        total_amount=('total_amount_mes', 'sum'),
        total_pedidos=('num_pedidos_mes', 'sum'),
        total_customer=('customer_id', 'nunique'),
    )
    .round(2)
)

merged = (
    total_amount_total
    .merge(total_amount_f, on=['order_created_month', 'is_target'], how='left', suffixes=('_all', '_out'))
    .assign(
        pct_amount_out=lambda x: (x['total_amount_out'] / x['total_amount_all']).round(4),
        pct_pedidos_out=lambda x: (x['total_pedidos_out'] / x['total_pedidos_all']).round(4),
        pct_c=lambda x: (x['total_customer_out'] / x['total_customer_all']).round(4)
    )
)

merged

In [ ]:
# CORREÇÃO: Filtro com parênteses corretos
total_amount_f= (
    publico_janeiro_dezembro[
        (publico_janeiro_dezembro['num_pedidos_mes'] >= 11) #& 
        #(publico_janeiro_dezembro['outlier'] == False)
    ]
    .groupby(['order_created_month', 'is_target'])
    .agg(
        total_amount=('total_amount_mes', 'sum'),
        total_pedidos=('num_pedidos_mes', 'sum'),
        total_customer=('customer_id', 'nunique'),

    )
    .round(2)
)

total_amount_total = (
    publico_janeiro_dezembro.groupby(['order_created_month', 'is_target'])
    .agg(
        total_amount=('total_amount_mes', 'sum'),
        total_pedidos=('num_pedidos_mes', 'sum'),
        total_customer=('customer_id', 'nunique'),
    )
    .round(2)
)

merged = (
    total_amount_total
    .merge(total_amount_f, on=['order_created_month', 'is_target'], how='left', suffixes=('_all', '_out'))
    .assign(
        pct_amount_out=lambda x: (x['total_amount_out'] / x['total_amount_all']).round(4),
        pct_pedidos_out=lambda x: (x['total_pedidos_out'] / x['total_pedidos_all']).round(4),
        pct_c=lambda x: (x['total_customer_out'] / x['total_customer_all']).round(4)
    )
)

merged

In [ ]:

pedidos_hist=resumo_coorte_ativa(publico_janeiro_dezembro,mes_coorte_inicio=12,mes_coorte_fim=1)
pedidos_hist

In [ ]:
import os

if os.path.exists('../Resultados'):
    pedidos_hist.to_csv('../Resultados/pedidos_hist.csv', index=False)
else:
    # Criar pasta ou salvar em outro lugar
    os.makedirs('../Resultados', exist_ok=True)
    pedidos_hist.to_csv('../Resultados/pedidos_hist.csv', index=False)
    print("Pasta criada e arquivo salvo!")

In [ ]:
pedidos_hist.to_csv('../Resultados/pedidos_hist.csv', index=False)
#df_stats_hist
#df_stats_mes

In [ ]:
with pd.ExcelWriter('../../Resultados/analise_pedidos.xlsx') as writer:
    pedidos_hist.to_excel(writer, sheet_name='Pedidos_Hist', index=False)
    df_stats_hist.to_excel(writer, sheet_name='Stats_Hist', index=True)
    df_stats_mes.to_excel(writer, sheet_name='Stats_Mes', index=True)